# Practical 4A — CIFAR-10 CNN Hyperparameter Study

## Aim
Design a basic CNN and study kernel sizes, strides, padding and pooling methods.

## Setup
Run the installation cell if required. These notebooks use Python 3.10–3.12 where possible. On Apple Silicon, TensorFlow/PyTorch acceleration depends on the installed build; CPU execution is supported.

**Important:** dataset downloads and model training may take time. Results are generated when you run the notebook; no accuracy values are pre-filled.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn tensorflow

## Reproducibility\nA fixed random seed is used where practical. If a dataset download fails, check your internet connection and rerun the loading cell.

## 1. Load CIFAR-10

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
np.random.seed(42); tf.random.set_seed(42)
(x_train,y_train),(x_test,y_test)=keras.datasets.cifar10.load_data()
x_train=x_train.astype("float32")/255.; x_test=x_test.astype("float32")/255.
y_train=y_train.ravel(); y_test=y_test.ravel()
# Subset option keeps repeated architecture experiments practical on a laptop.
xtr,ytr=x_train[:20000],y_train[:20000]

## 2. CNN builder

In [ ]:
def build_cnn(kernel=3,stride=1,padding="same",pool="max"):
    Pool=layers.MaxPooling2D if pool=="max" else layers.AveragePooling2D
    model=keras.Sequential([
        layers.Input((32,32,3)),
        layers.Conv2D(32,kernel,strides=stride,padding=padding,activation="relu"),
        Pool(2), layers.Conv2D(64,3,padding="same",activation="relu"), Pool(2),
        layers.Flatten(),layers.Dense(128,activation="relu"),layers.Dense(10,activation="softmax")])
    model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model

## 3. Run controlled architecture experiments

In [ ]:
configs=[
 ("k3_s1_same_max",3,1,"same","max"),
 ("k5_s1_same_max",5,1,"same","max"),
 ("k3_s2_same_max",3,2,"same","max"),
 ("k3_s1_valid_max",3,1,"valid","max"),
 ("k3_s1_same_avg",3,1,"same","avg")]
results=[]
for name,k,s,p,pool in configs:
    m=build_cnn(k,s,p,pool); start=time.time()
    h=m.fit(xtr,ytr,validation_split=0.1,epochs=3,batch_size=128,verbose=0)
    loss,acc=m.evaluate(x_test[:5000],y_test[:5000],verbose=0)
    results.append((name,acc,loss,time.time()-start))
    print(name,"accuracy",acc,"loss",loss)

In [ ]:
import pandas as pd
df=pd.DataFrame(results,columns=["Configuration","Test accuracy (5k subset)","Test loss","Training seconds"])
display(df)
df.plot(x="Configuration",y="Test accuracy (5k subset)",kind="bar",legend=False,rot=25); plt.tight_layout(); plt.show()

## 4. Conclusion\nThe experiment compares kernel size, stride, padding and pooling. For a fair comparison keep epochs, data subset, optimizer and batch size fixed. Accuracy is evaluated on a 5,000-image test subset to reduce runtime.